# 🛒 Product Review Intelligence System
### BERT / DeBERTa Aspect-Based Sentiment Analysis on a Kaggle Dataset — Complete Google Colab Pipeline

This notebook builds the **project idea** from the classroom ABSA notebook, end to end, on real Kaggle reviews:

| Requirement | Where it happens |
|---|---|
| ✅ accepts customer reviews | **Stage 1** — download a Kaggle dataset, clean and sample reviews |
| ✅ extracts candidate aspects | **Stage 3** — spaCy noun phrases + corpus frequency filtering |
| ✅ predicts aspect-level sentiment | **Stage 4** — `sentence + aspect → sentiment` with a DeBERTa ABSA model |
| ✅ calculates confidence | **Stage 5** — probability, margin and entropy combined into one score |
| ✅ visualizes positive and negative aspects | **Stage 7** — diverging bars, priority map, product heatmap |
| ✅ produces a dashboard | **Stage 9** — a single HTML dashboard you can open, download and submit |

**The pipeline**

```text
Kaggle CSV → Clean → Sentences → Candidate aspects → ABSA model → Confidence → Aggregate → Charts → Dashboard
```

**Before you start:** `Runtime → Change runtime type → T4 GPU` (CPU works too, just slower).
Then run every cell from top to bottom (**Runtime → Run all** works).

In [ ]:
# CELL 1 — Safe Google Colab setup (no restart needed)
# ============================================================
# Colab already ships PyTorch, Transformers, spaCy, pandas and matplotlib that are
# built to work together. Re-installing or pinning them (especially torch or
# transformers) is what causes errors such as:
#   "module 'torch' has no attribute '_utils'"
#   "operator torchvision::nms does not exist"
# So this cell ONLY installs what is missing and never touches torch.
# ============================================================

import importlib.util
import subprocess
import sys

def is_missing(module):
    try:
        return importlib.util.find_spec(module) is None
    except ModuleNotFoundError:
        return True

needed = {
    "kagglehub": "kagglehub",
    "transformers": "transformers",
    "sentencepiece": "sentencepiece",      # DeBERTa-v3 tokenizer
    "google.protobuf": "protobuf",         # DeBERTa-v3 tokenizer
    "spacy": "spacy",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
}
to_install = [pkg for module, pkg in needed.items() if is_missing(module)]

if to_install:
    print("Installing:", ", ".join(to_install))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *to_install])
else:
    print("All Python packages are already available.")

if is_missing("en_core_web_sm"):
    print("Installing spaCy English model...")
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"])

print()
print("✅ Setup complete. Continue with Cell 2 (no restart needed).")
print("   If you ran an older Cell 1 that pinned transformers/torch in this session:")
print("   Runtime → Disconnect and delete runtime, then run this notebook from the top.")

In [ ]:
# CELL 2 — Import and verify everything
# ============================================================

import os, re, sys, glob, html, json, math, time, base64, shutil, warnings
from collections import Counter
from io import BytesIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
try:
    import torch
    import transformers
    from transformers import pipeline
except Exception as e:
    raise RuntimeError(
        "PyTorch/Transformers failed to import. This happens when packages were re-installed "
        "in a running session. Fix: Runtime → Disconnect and delete runtime, then run Cell 1 "
        f"and continue from Cell 2.\n\nOriginal error: {e}") from e
from tqdm.auto import tqdm
from IPython.display import display, HTML

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.05)
pd.set_option("display.max_colwidth", 120)

# One colour per sentiment, used in every chart
POS_COLOR, NEG_COLOR, NEU_COLOR = "#2E8B57", "#C0392B", "#95A5A6"
PALETTE = {"Positive": POS_COLOR, "Negative": NEG_COLOR, "Neutral": NEU_COLOR}

DEVICE = 0 if torch.cuda.is_available() else -1
nlp = spacy.load("en_core_web_sm")

print("✅ Environment loaded successfully!\n")
print("Python       :", sys.version.split()[0])
print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("spaCy        :", spacy.__version__)
print("Device       :", "GPU 🚀" if DEVICE == 0 else "CPU 🐢 (works, but use a small sample)")

In [ ]:
# CELL 3 — Project configuration (change these, nothing else)
# ============================================================

# Any Kaggle review dataset with a text column works. Good choices:
#   "datafiniti/consumer-reviews-of-amazon-products"  (Amazon devices, ~70k reviews)  ← default
#   "snap/amazon-fine-food-reviews"                    (food products, 568k reviews, large download)
KAGGLE_DATASET = "datafiniti/consumer-reviews-of-amazon-products"

# The ABSA model runs once per (sentence, aspect) pair, so we analyse a sample.
SAMPLE_SIZE     = 1500 if torch.cuda.is_available() else 200
BALANCE_RATINGS = True     # sample evenly across star ratings so complaints are not drowned out
RANDOM_SEED     = 42

MIN_ASPECT_REVIEWS = 5     # an aspect must appear in at least this many reviews
MAX_ASPECTS        = 30    # keep the most frequent aspects
MIN_MENTIONS_SHOWN = 5     # hide aspects with fewer mentions from the charts
BATCH_SIZE         = 32

ABSA_MODEL     = "yangheng/deberta-v3-base-absa-v1.1"                 # sentence + aspect → sentiment
DOCUMENT_MODEL = "distilbert-base-uncased-finetuned-sst-2-english"    # whole review → sentiment

OUTPUT_DIR = "review_intelligence_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
np.random.seed(RANDOM_SEED)

print("Dataset     :", KAGGLE_DATASET)
print("Sample size :", SAMPLE_SIZE)
print("Outputs     :", os.path.abspath(OUTPUT_DIR))

# 📥 STAGE 1 — Accept customer reviews

We download the dataset straight from Kaggle with `kagglehub`. Public datasets usually download without logging in.

If the download fails, Cell 5 lets you **upload the CSV** yourself (download it from the Kaggle page first).
If even that is skipped, a small built-in demo set is used so the notebook still runs.

In [ ]:
# CELL 4 — Download the Kaggle dataset
# ============================================================

def download_kaggle_dataset(dataset):
    try:
        import kagglehub
        path = kagglehub.dataset_download(dataset)
        csvs = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
        csvs = sorted(csvs, key=os.path.getsize, reverse=True)
        print("✅ Downloaded to:", path)
        for f in csvs:
            print(f"   {os.path.basename(f):70s} {os.path.getsize(f) / 1e6:7.1f} MB")
        return csvs
    except Exception as e:
        print("⚠️ Kaggle download failed:", e)
        print("   Use Cell 5 to upload the CSV instead.")
        return []

csv_files = download_kaggle_dataset(KAGGLE_DATASET)

In [ ]:
# CELL 5 — (Only if Cell 4 failed) upload a CSV from your computer
# ============================================================

if not csv_files:
    try:
        from google.colab import files
        print("📤 Choose the review CSV you downloaded from Kaggle:")
        uploaded = files.upload()
        csv_files = list(uploaded.keys())
    except Exception as e:
        print("Upload skipped:", e)

if not csv_files:
    print("ℹ️ No CSV available — the built-in demo reviews will be used.")

In [ ]:
# CELL 6 — Load, standardise, clean and sample the reviews
# ============================================================
# Every Kaggle dataset names its columns differently, so we detect them.

TEXT_CANDIDATES    = ["reviews.text", "Text", "text", "review", "Review", "review_text",
                      "reviewText", "content", "Review Text", "comment", "body"]
RATING_CANDIDATES  = ["reviews.rating", "Score", "rating", "Rating", "stars", "overall", "score"]
PRODUCT_CANDIDATES = ["name", "ProductId", "product", "product_name", "Product Name", "asin", "brand"]

DEMO_REVIEWS = [
    ("The screen is gorgeous but the battery life is disappointing.", 3, "Demo Tablet"),
    ("Setup was easy and the sound quality is excellent for the price.", 5, "Demo Speaker"),
    ("The speakers are tinny and Alexa often misunderstands me.", 2, "Demo Speaker"),
    ("Great price, fast delivery, but the charger stopped working after a week.", 3, "Demo Tablet"),
    ("The camera is terrible and the tablet is really slow.", 1, "Demo Tablet"),
    ("I love the display and the battery lasts for days.", 5, "Demo E-reader"),
    ("The remote is cheap and the picture quality is only okay.", 3, "Demo TV Stick"),
    ("The picture quality is stunning and the interface is fast.", 5, "Demo TV Stick"),
    ("Customer service was rude and the replacement took weeks.", 1, "Demo E-reader"),
    ("The light weight makes reading comfortable, and the screen has no glare.", 5, "Demo E-reader"),
    ("The battery drains quickly and the screen is too dim.", 2, "Demo Tablet"),
    ("Sound is decent for the size but the price is too high.", 3, "Demo Speaker"),
]

def pick_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def clean_text(text):
    text = html.unescape(str(text))
    text = re.sub(r"<[^>]+>", " ", text)          # HTML tags such as <br />
    text = re.sub(r"https?://\S+", " ", text)     # URLs
    return re.sub(r"\s+", " ", text).strip()

def short_product_name(name, max_len=42):
    name = str(name)
    name = re.split(r"\s[-–]\s|,|\(", name)[0].strip()
    return name if len(name) <= max_len else name[: max_len - 1] + "…"

def load_reviews(csv_files, sample_size, balance=True, seed=42):
    frames = []
    for f in csv_files:
        try:
            df = pd.read_csv(f, low_memory=False, on_bad_lines="skip")
        except Exception as e:
            print("Skipping", f, "-", e)
            continue
        tcol = pick_column(df, TEXT_CANDIDATES)
        if tcol is None:
            continue
        rcol, pcol = pick_column(df, RATING_CANDIDATES), pick_column(df, PRODUCT_CANDIDATES)
        print(f"📄 {os.path.basename(f)}: text='{tcol}', rating='{rcol}', product='{pcol}' ({len(df):,} rows)")
        frames.append(pd.DataFrame({
            "text": df[tcol],
            "rating": pd.to_numeric(df[rcol], errors="coerce") if rcol else np.nan,
            "product": df[pcol] if pcol else "All products",
        }))

    if frames:
        data = pd.concat(frames, ignore_index=True)
        source = "Kaggle"
    else:
        data = pd.DataFrame(DEMO_REVIEWS, columns=["text", "rating", "product"])
        source = "demo"

    data["text"] = data["text"].map(clean_text)
    data["product"] = data["product"].fillna("Unknown product").map(short_product_name)
    data = data[data["text"].str.len() >= 20].drop_duplicates("text")
    total = len(data)

    # Sample: evenly across star ratings (so complaints are visible) or purely at random
    if len(data) > sample_size:
        if balance and data["rating"].notna().any():
            per_star = sample_size // data["rating"].nunique()
            parts = [g.sample(min(len(g), per_star), random_state=seed) for _, g in data.groupby("rating")]
            sampled = pd.concat(parts)
            if len(sampled) < sample_size:            # top up when some star groups are small
                rest = data.drop(sampled.index)
                sampled = pd.concat([sampled, rest.sample(min(len(rest), sample_size - len(sampled)),
                                                          random_state=seed)])
            data = sampled.sample(frac=1, random_state=seed)
        else:
            data = data.sample(sample_size, random_state=seed)

    data = data.reset_index(drop=True)
    data.insert(0, "review_id", range(len(data)))
    print(f"\n✅ Using {len(data):,} of {total:,} clean reviews (source: {source})")
    return data

reviews = load_reviews(csv_files, SAMPLE_SIZE, BALANCE_RATINGS, RANDOM_SEED)
display(reviews.head(8))

In [ ]:
# CELL 7 — Explore the data before analysing it
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

if reviews["rating"].notna().any():
    sns.countplot(x=reviews["rating"].dropna().astype(int), ax=axes[0], color="#4C72B0")
axes[0].set_title("Star ratings in the sample")
axes[0].set_xlabel("Stars")

lengths = reviews["text"].str.split().str.len()
sns.histplot(lengths.clip(upper=200), bins=40, ax=axes[1], color="#4C72B0")
axes[1].set_title(f"Review length (median {int(lengths.median())} words)")
axes[1].set_xlabel("Words (clipped at 200)")

top_products = reviews["product"].value_counts().head(10)
sns.barplot(x=top_products.values, y=top_products.index, ax=axes[2], color="#4C72B0")
axes[2].set_title("Most-reviewed products")
axes[2].set_xlabel("Reviews")

plt.tight_layout()
plt.show()

# ✂️ STAGE 2 — Preprocess: reviews → sentences

A long review usually contains several opinions:

> *"The screen is gorgeous. Sadly the battery dies by lunchtime."*

We give the ABSA model **one sentence at a time** together with the aspect. That keeps the context
focused on the opinion about that aspect, exactly like `"I love chicken but hate KFC." + "KFC"`.

spaCy also tags every word with its part of speech, which Stage 3 uses to find aspects.

In [ ]:
# CELL 8 — Run spaCy on every review
# ============================================================

t0 = time.time()
docs = list(tqdm(nlp.pipe(reviews["text"].tolist(), batch_size=64), total=len(reviews), desc="spaCy"))
n_sentences = sum(1 for d in docs for _ in d.sents)

print(f"✅ Processed {len(docs):,} reviews into {n_sentences:,} sentences in {time.time() - t0:.1f}s")
print("\nExample review split into sentences:")
example_doc = max(docs[:50], key=lambda d: len(list(d.sents)))
for i, s in enumerate(example_doc.sents, 1):
    print(f"  {i}. {s.text}")

# 🔎 STAGE 3 — Extract candidate aspects

The classroom notebook used *every noun* as an aspect. On thousands of real reviews that gives lots of noise
("thing", "time", "Amazon", "husband"…). We improve it in four steps:

1. **Noun phrases** — spaCy noun chunks keep phrases like *battery life* and *picture quality* together.
2. **Lemmas** — *batteries* and *battery* become one aspect.
3. **Filter generic words** — remove words that are never a product feature.
4. **Corpus frequency** — keep only aspects that appear in at least `MIN_ASPECT_REVIEWS` different reviews.

Synonyms are merged into one name (*display → screen*, *cost → price*).

In [ ]:
# CELL 9 — Visualise candidate extraction on one sentence (same idea as the classroom notebook)
# ============================================================

demo_sentence = "The camera takes amazing photos, but the battery life is short and the screen is too dim."
doc = nlp(demo_sentence)

fig, ax = plt.subplots(figsize=(16, 3))
ax.axis("off")
for i, token in enumerate(doc):
    is_candidate = token.pos_ in ("NOUN", "PROPN")
    ax.text(i, 0.6, token.text, ha="center", fontsize=13,
            bbox=dict(boxstyle="round,pad=0.35", edgecolor="black",
                      facecolor="lightgreen" if is_candidate else "white"))
    ax.text(i, 0.25, token.pos_, ha="center", fontsize=10, color="dimgray")
ax.set_xlim(-1, len(doc))
ax.set_ylim(0, 1)
ax.set_title("POS tags — green = noun candidates", fontsize=15)
plt.show()

print("Noun chunks found by spaCy:", [chunk.text for chunk in doc.noun_chunks])

In [ ]:
# CELL 10 — Aspect extraction rules
# ============================================================

GENERIC_WORDS = {
    # never product features
    "thing", "things", "lot", "time", "times", "day", "days", "week", "weeks", "month", "months",
    "year", "years", "way", "one", "ones", "bit", "kind", "sort", "reason", "problem", "issue",
    "product", "item", "purchase", "buy", "order", "review", "star", "stars", "amazon", "money",
    "everything", "anything", "something", "nothing", "everyone", "people", "person", "use", "fun",
    "gift", "christmas", "birthday", "husband", "wife", "son", "daughter", "kid", "kids", "child",
    "children", "mom", "dad", "family", "friend", "friends", "grandson", "granddaughter", "home",
    "house", "room", "job", "deal", "need", "hour", "hours", "minute", "minutes",
    # the products themselves (we want their features, not the product name)
    "device", "devices", "tablet", "tablets", "kindle", "echo", "fire", "unit", "model", "version",
}

ASPECT_SYNONYMS = {
    "price": "price", "cost": "price", "value": "price", "pricing": "price",
    "battery": "battery", "battery life": "battery", "charge": "battery",
    "screen": "screen", "display": "screen", "resolution": "screen",
    "sound": "sound", "sound quality": "sound", "audio": "sound", "speaker": "sound", "volume": "sound",
    "picture": "picture quality", "picture quality": "picture quality", "image quality": "picture quality",
    "camera": "camera", "photo": "camera",
    "app": "apps", "apps": "apps", "application": "apps",
    "setup": "setup", "set up": "setup", "installation": "setup",
    "customer service": "customer service", "support": "customer service", "service": "customer service",
    "shipping": "delivery", "delivery": "delivery",
    "wifi": "connectivity", "wi fi": "connectivity", "bluetooth": "connectivity", "connection": "connectivity",
    "alexa": "alexa", "voice": "alexa",
    "design": "design", "look": "design", "size": "size", "weight": "weight",
    "speed": "speed", "performance": "speed", "processor": "speed",
    "charger": "charger", "cable": "charger", "remote": "remote", "keyboard": "keyboard",
    "case": "case", "cover": "case", "storage": "storage", "memory": "storage",
    "quality": "build quality", "build quality": "build quality", "build": "build quality",
}

def chunk_to_aspect(chunk):
    """Turn a spaCy noun chunk into (canonical_aspect, surface_text) or None."""
    nouns = [t for t in chunk if t.pos_ in ("NOUN", "PROPN") and t.is_alpha and len(t.text) > 2]
    if not nouns:
        return None
    nouns = nouns[-2:]                                  # keep up to two words: "battery life"
    lemmas = [t.lemma_.lower() for t in nouns]
    phrase = " ".join(lemmas)
    surface = " ".join(t.text for t in nouns)

    if phrase in ASPECT_SYNONYMS:
        return ASPECT_SYNONYMS[phrase], surface
    if lemmas[-1] in ASPECT_SYNONYMS:
        return ASPECT_SYNONYMS[lemmas[-1]], nouns[-1].text
    if lemmas[-1] in GENERIC_WORDS:
        return None
    if len(lemmas) == 2 and lemmas[0] in GENERIC_WORDS:   # "kindle screen" → "screen"
        return lemmas[-1], nouns[-1].text
    return phrase, surface

def extract_candidates(doc):
    """All candidate aspects in one review, one row per (sentence, aspect)."""
    rows = []
    for sent_idx, sent in enumerate(doc.sents):
        seen = set()
        for chunk in sent.noun_chunks:
            result = chunk_to_aspect(chunk)
            if result and result[0] not in seen:
                seen.add(result[0])
                rows.append({"sent_idx": sent_idx, "sentence": sent.text.strip(),
                             "aspect": result[0], "surface": result[1]})
    return rows

print("Test:", extract_candidates(nlp(demo_sentence)))

In [ ]:
# CELL 11 — Extract candidates from every review, then keep the frequent ones
# ============================================================

candidate_rows = []
for review_id, doc in zip(reviews["review_id"], docs):
    for row in extract_candidates(doc):
        row["review_id"] = review_id
        candidate_rows.append(row)
candidates = pd.DataFrame(candidate_rows)

review_freq = candidates.groupby("aspect")["review_id"].nunique().sort_values(ascending=False)
min_reviews = min(MIN_ASPECT_REVIEWS, max(2, len(reviews) // 20))   # relax the rule for tiny datasets
kept_aspects = review_freq[review_freq >= min_reviews].head(MAX_ASPECTS).index.tolist()

mentions = (candidates[candidates["aspect"].isin(kept_aspects)]
            .merge(reviews[["review_id", "rating", "product"]], on="review_id"))

print(f"Candidate aspects found : {candidates['aspect'].nunique():,}")
print(f"Aspects kept            : {len(kept_aspects)}")
print(f"(sentence, aspect) pairs: {len(mentions):,}  ← these go to the ABSA model")
print(f"Reviews with an aspect  : {mentions['review_id'].nunique():,} / {len(reviews):,}")

# Show what was kept and what was cut
top = review_freq.head(MAX_ASPECTS + 15)
colors = ["#4C72B0" if a in kept_aspects else "#D0D0D0" for a in top.index]
plt.figure(figsize=(10, 0.28 * len(top) + 1))
plt.barh(top.index[::-1], top.values[::-1], color=colors[::-1])
plt.axvline(min_reviews, color="black", ls=":", lw=1)
plt.title("Candidate aspects (blue = kept, grey = cut)")
plt.xlabel("Number of reviews mentioning the aspect")
plt.tight_layout()
plt.show()

# 🧠 STAGE 4 — Predict aspect-level sentiment

Same idea as the classroom notebook:

**Sentence + Aspect → Aspect Sentiment**

We send all pairs to the ABSA model in batches and keep the probability of **all three** classes
(Negative / Neutral / Positive), not just the winner. Stage 5 needs them to compute confidence.

In [ ]:
# CELL 12 — Load the ABSA model and sanity-check it
# ============================================================

absa_pipeline = pipeline("text-classification", model=ABSA_MODEL, tokenizer=ABSA_MODEL, device=DEVICE)

# Input format: {"text": sentence, "text_pair": aspect}  →  Sentence + Aspect → Sentiment
for aspect in ["chicken", "KFC"]:
    out = absa_pipeline({"text": "I love chicken but hate KFC.", "text_pair": aspect}, top_k=None)
    out = out if isinstance(out, list) else [out]
    best = max(out, key=lambda p: p["score"])
    print(f"I love chicken but hate KFC.  +  {aspect:8s} → {best['label']:8s} ({best['score']:.2%})")

In [ ]:
# CELL 13 — Run ABSA on every (sentence, aspect) pair
# ============================================================

def probabilities(prediction):
    """Turn the pipeline output into {'Negative': p, 'Neutral': p, 'Positive': p}."""
    if isinstance(prediction, dict):
        prediction = [prediction]
    probs = {"Negative": 0.0, "Neutral": 0.0, "Positive": 0.0}
    for p in prediction:
        probs[p["label"].capitalize()] = float(p["score"])
    return probs

def predict_aspect_sentiment(sentences, aspects, batch_size=BATCH_SIZE):
    inputs = [{"text": s, "text_pair": a} for s, a in zip(sentences, aspects)]
    if not inputs:
        return pd.DataFrame(columns=["Negative", "Neutral", "Positive"], dtype=float)
    results = []
    step = batch_size * 4
    for start in tqdm(range(0, len(inputs), step), desc="ABSA"):
        chunk = inputs[start:start + step]
        out = absa_pipeline(chunk, top_k=None, batch_size=batch_size, truncation=True, max_length=256)
        results.extend(probabilities(p) for p in out)
    return pd.DataFrame(results)

t0 = time.time()
probs = predict_aspect_sentiment(mentions["sentence"].tolist(), mentions["surface"].tolist())
mentions[["p_negative", "p_neutral", "p_positive"]] = probs[["Negative", "Neutral", "Positive"]].values
mentions["sentiment"] = probs.idxmax(axis=1).values

elapsed = time.time() - t0
print(f"✅ {len(mentions):,} aspect predictions in {elapsed:.0f}s ({len(mentions) / max(elapsed, 1e-9):.0f} pairs/s)")
display(mentions[["sentence", "aspect", "sentiment", "p_negative", "p_neutral", "p_positive"]].head(10))

# 📊 STAGE 5 — Calculate confidence

The classroom notebook used the **top probability** as confidence. That is a good start, but two predictions
with the same top probability can be very different:

| Probabilities (Neg / Neu / Pos) | Top prob | What's really happening |
|---|---|---|
| 0.05 / 0.35 / **0.60** | 0.60 | clearly not negative, unsure between neutral and positive |
| 0.38 / 0.02 / **0.60** | 0.60 | torn between *opposite* opinions — much less trustworthy |

So we combine three signals:

- **Probability** – the top class probability
- **Margin** – gap between the top two classes (bigger gap = clearer decision)
- **Certainty** – `1 − entropy / log(3)`, how concentrated the whole distribution is

`confidence = 0.5 × probability + 0.3 × margin + 0.2 × certainty`

We also compute a **sentiment score** `= P(positive) − P(negative)`, from −1 (very negative) to +1 (very positive).

In [ ]:
# CELL 14 — Confidence scores
# ============================================================

def add_confidence(df):
    P = df[["p_negative", "p_neutral", "p_positive"]].to_numpy()
    top2 = np.sort(P, axis=1)[:, -2:]
    entropy = -(P * np.log(P + 1e-12)).sum(axis=1) / np.log(3)

    df["probability"] = top2[:, 1]
    df["margin"] = top2[:, 1] - top2[:, 0]
    df["certainty"] = 1 - entropy
    df["confidence"] = (0.5 * df["probability"] + 0.3 * df["margin"] + 0.2 * df["certainty"]).clip(0, 1)
    df["confidence_band"] = pd.cut(df["confidence"], [-0.01, 0.5, 0.8, 1.0], labels=["Low", "Medium", "High"])
    df["score"] = df["p_positive"] - df["p_negative"]
    return df

mentions = add_confidence(mentions)

print("Confidence bands:")
print(mentions["confidence_band"].value_counts().reindex(["High", "Medium", "Low"]).to_string())

cols = ["sentence", "aspect", "sentiment", "probability", "margin", "confidence"]
print("\n🟢 Most confident predictions")
display(mentions.sort_values("confidence", ascending=False)[cols].head(5).round(3))
print("🟠 Least confident predictions (worth checking by hand)")
display(mentions.sort_values("confidence")[cols].head(5).round(3))

In [ ]:
# CELL 15 — Confidence distribution
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))

sns.histplot(data=mentions, x="confidence", hue="sentiment", palette=PALETTE,
             bins=25, multiple="stack", ax=axes[0])
for x in (0.5, 0.8):
    axes[0].axvline(x, color="black", ls=":", lw=1)
axes[0].set_title("How sure is the model? (dotted lines = Low | Medium | High)")

sns.scatterplot(data=mentions, x="probability", y="margin", hue="sentiment",
                palette=PALETTE, alpha=0.5, s=18, ax=axes[1])
axes[1].set_title("Top probability vs margin")

plt.tight_layout()
plt.show()

# 🧮 STAGE 6 — Aggregate: from thousands of predictions to a product report

For every aspect we count positive, negative and neutral mentions and average the confidence.

**Ranking fairly.** An aspect with 3 / 3 positive mentions should not beat one with 480 / 500.
We use the **Wilson lower bound** — the lowest positive share we can be 95% sure of, given how many
mentions we have. An aspect is called a **strength** or **weakness** only when that bound passes 50%.

In [ ]:
# CELL 16 — Aspect summary table
# ============================================================

def wilson_lower_bound(successes, n, z=1.96):
    if n == 0:
        return 0.0
    p = successes / n
    centre = p + z * z / (2 * n)
    margin = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (centre - margin) / (1 + z * z / n)

def summarise_aspects(m, min_mentions=MIN_MENTIONS_SHOWN):
    g = m.groupby("aspect")
    s = pd.DataFrame({
        "mentions": g.size(),
        "positive": g["sentiment"].apply(lambda x: (x == "Positive").sum()),
        "negative": g["sentiment"].apply(lambda x: (x == "Negative").sum()),
        "neutral": g["sentiment"].apply(lambda x: (x == "Neutral").sum()),
        "avg_score": g["score"].mean(),
        "avg_confidence": g["confidence"].mean(),
    })
    s["weighted_score"] = m.groupby("aspect").apply(
        lambda x: np.average(x["score"], weights=x["confidence"] + 1e-6), include_groups=False)
    polar = s["positive"] + s["negative"]
    s["positive_share"] = (s["positive"] / polar.replace(0, np.nan)).fillna(0.5)
    s["pos_lower"] = [wilson_lower_bound(p, n) for p, n in zip(s["positive"], polar)]
    s["neg_lower"] = [wilson_lower_bound(q, n) for q, n in zip(s["negative"], polar)]
    s["verdict"] = np.select([s["pos_lower"] >= 0.5, s["neg_lower"] >= 0.5],
                             ["💪 Strength", "⚠️ Weakness"], default="⚖️ Mixed")
    return s[s["mentions"] >= min_mentions].sort_values("mentions", ascending=False)

summary = summarise_aspects(mentions, min_mentions=min(MIN_MENTIONS_SHOWN, max(2, len(reviews) // 20)))
if summary.empty:
    print("⚠️ No aspect has enough mentions. Increase SAMPLE_SIZE or lower MIN_MENTIONS_SHOWN.")

display(summary[["mentions", "positive", "negative", "neutral", "positive_share",
                 "weighted_score", "avg_confidence", "verdict"]]
        .style.format({"positive_share": "{:.0%}", "weighted_score": "{:+.2f}", "avg_confidence": "{:.2f}"})
        .background_gradient(subset=["weighted_score"], cmap="RdYlGn", vmin=-1, vmax=1)
        .bar(subset=["mentions"], color="#C8D6E5"))

In [ ]:
# CELL 17 — Top strengths and weaknesses with evidence
# ============================================================

def best_evidence(aspect, sentiment, k=2):
    rows = mentions[(mentions["aspect"] == aspect) & (mentions["sentiment"] == sentiment)]
    rows = rows.sort_values("confidence", ascending=False).drop_duplicates("sentence").head(k)
    return [f'"{r.sentence}" ({r.confidence:.2f})' for r in rows.itertuples()]

for title, col, sentiment in [("💪 TOP STRENGTHS", "pos_lower", "Positive"),
                              ("⚠️ TOP WEAKNESSES", "neg_lower", "Negative")]:
    print("=" * 90)
    print(title)
    print("=" * 90)
    for aspect, row in summary.sort_values(col, ascending=False).head(5).iterrows():
        share = row["positive_share"] if sentiment == "Positive" else 1 - row["positive_share"]
        print(f"\n▶ {aspect}: {share:.0%} {sentiment.lower()} of {int(row['positive'] + row['negative'])} opinions")
        for quote in best_evidence(aspect, sentiment):
            print("   ", quote)
    print()

# 🎨 STAGE 7 — Visualise positive and negative aspects

Four charts, each answering one question:

1. **What do customers praise and complain about?** — diverging bars
2. **What should the company fix first?** — priority map (frequent + negative = urgent)
3. **How does each product compare?** — product × aspect heatmap
4. **How many reviews are mixed?** — the "I love chicken but hate KFC" reviews

In [ ]:
# CELL 18 — Diverging bar chart: praise vs complaints
# ============================================================

def plot_diverging(summary, top=15, ax=None):
    s = summary.head(top).sort_values("weighted_score")
    ax = ax or plt.subplots(figsize=(10, 0.45 * len(s) + 1.5))[1]
    if s.empty:
        return ax
    y = np.arange(len(s))
    ax.barh(y, -s["negative"], color=NEG_COLOR, label="Negative")
    ax.barh(y, s["positive"], color=POS_COLOR, label="Positive")
    ax.barh(y, s["neutral"], left=s["positive"], color=NEU_COLOR, alpha=0.6, label="Neutral")
    ax.set_yticks(y, s.index)
    ax.axvline(0, color="black", lw=0.8)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{abs(v):.0f}"))
    ax.set_xlabel("Mentions   (← negative | positive →)")
    ax.set_title("What customers praise and complain about", fontweight="bold")
    ax.legend(loc="lower right")
    return ax

plot_diverging(summary)
plt.tight_layout()
plt.show()

In [ ]:
# CELL 19 — Priority map: what to fix first
# ============================================================

def plot_priority_map(summary, top=25, ax=None):
    s = summary.head(top)
    ax = ax or plt.subplots(figsize=(11, 6))[1]
    if s.empty:
        return ax
    colors = s["weighted_score"].map(lambda v: POS_COLOR if v > 0.15 else NEG_COLOR if v < -0.15 else NEU_COLOR)
    ax.scatter(s["mentions"], s["weighted_score"], s=60 + 400 * s["avg_confidence"],
               c=colors, alpha=0.8, edgecolors="white", linewidths=1.5)
    for aspect, r in s.iterrows():
        ax.annotate(aspect, (r["mentions"], r["weighted_score"]), xytext=(6, 4),
                    textcoords="offset points", fontsize=9)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xscale("log")
    ax.set_xlabel("How often it is mentioned (log scale)")
    ax.set_ylabel("Confidence-weighted sentiment")
    ax.set_title("Priority map — bottom right = frequent complaints = fix first", fontweight="bold")
    return ax

plot_priority_map(summary)
plt.tight_layout()
plt.show()

In [ ]:
# CELL 20 — Product × aspect heatmap
# ============================================================

def product_aspect_matrix(m, aspects, top_products=10, min_count=3):
    products = m["product"].value_counts().head(top_products).index
    sub = m[m["product"].isin(products) & m["aspect"].isin(aspects)]
    mean = sub.pivot_table(index="product", columns="aspect", values="score", aggfunc="mean")
    count = sub.pivot_table(index="product", columns="aspect", values="score", aggfunc="count")
    return mean.where(count >= min_count)          # hide cells with too little evidence

matrix = product_aspect_matrix(mentions, summary.head(12).index)

if matrix.shape[0] >= 2 and matrix.notna().any().any():
    plt.figure(figsize=(min(1.1 * matrix.shape[1] + 4, 20), 0.55 * matrix.shape[0] + 2))
    sns.heatmap(matrix, annot=True, fmt="+.2f", cmap="RdYlGn", vmin=-1, vmax=1,
                linewidths=0.5, cbar_kws={"label": "Average aspect sentiment"})
    plt.title("Aspect sentiment by product (blank = fewer than 3 mentions)", fontweight="bold")
    plt.ylabel("")
    plt.xlabel("")
    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ Only one product in the data, so there is nothing to compare.")

In [ ]:
# CELL 21 — Mixed reviews: why ABSA beats one overall label
# ============================================================

per_review = mentions[mentions["confidence_band"] != "Low"].groupby("review_id")["sentiment"]
has_pos = per_review.apply(lambda x: (x == "Positive").any())
has_neg = per_review.apply(lambda x: (x == "Negative").any())

review_type = pd.Series("Only neutral", index=has_pos.index)
review_type[has_pos & ~has_neg] = "Only positive aspects"
review_type[~has_pos & has_neg] = "Only negative aspects"
review_type[has_pos & has_neg] = "Mixed (positive AND negative)"
reviews["review_type"] = reviews["review_id"].map(review_type).fillna("No aspect found")

counts = reviews["review_type"].value_counts()
plt.figure(figsize=(9, 4))
sns.barplot(x=counts.values, y=counts.index,
            palette=[{"Only positive aspects": POS_COLOR, "Only negative aspects": NEG_COLOR,
                      "Mixed (positive AND negative)": "#E6A23C"}.get(i, NEU_COLOR) for i in counts.index])
plt.title(f"{(review_type.str.startswith('Mixed')).sum()} reviews contain opposite opinions", fontweight="bold")
plt.xlabel("Reviews")
plt.tight_layout()
plt.show()

# ⚖️ STAGE 8 — Standard BERT vs ABSA on real reviews

The classroom notebook showed this on one sentence. Now we check it on the whole dataset:

- **Standard BERT** gives each review one label.
- **ABSA** gives each aspect its own label.

Kaggle data has **star ratings but no aspect labels**, so we use stars as a rough reference
(4–5 ★ = positive, 1–2 ★ = negative, 3 ★ skipped). We also check whether high-confidence ABSA
predictions agree with the stars more often than low-confidence ones — if they do, our confidence score is meaningful.

In [ ]:
# CELL 22 — Document-level BERT on every review
# ============================================================

document_pipeline = pipeline("sentiment-analysis", model=DOCUMENT_MODEL, device=DEVICE)

doc_preds = []
for start in tqdm(range(0, len(reviews), 256), desc="BERT"):
    batch = reviews["text"].iloc[start:start + 256].tolist()
    doc_preds.extend(document_pipeline(batch, batch_size=BATCH_SIZE, truncation=True, max_length=512))

reviews["bert_label"] = [p["label"].capitalize() for p in doc_preds]
reviews["bert_confidence"] = [p["score"] for p in doc_preds]

print("\n🍗 Mixed reviews: one BERT label vs several ABSA labels\n")
mixed_ids = reviews.loc[reviews["review_type"].str.startswith("Mixed"), "review_id"].head(4)
for rid in mixed_ids:
    r = reviews.loc[reviews["review_id"] == rid].iloc[0]
    print("📝", r["text"][:300] + ("…" if len(r["text"]) > 300 else ""))
    print(f"   Standard BERT → {r['bert_label']} ({r['bert_confidence']:.0%})")
    asp = mentions[mentions["review_id"] == rid][["aspect", "sentiment", "confidence"]]
    print("   ABSA          →", ", ".join(f"{a}: {s} ({c:.2f})" for a, s, c in asp.itertuples(index=False)))
    print("-" * 100)

In [ ]:
# CELL 23 — Evaluation against star ratings
# ============================================================

from sklearn.metrics import accuracy_score, f1_score

evaluation = {}
labelled = reviews.dropna(subset=["rating"])
labelled = labelled[labelled["rating"] != 3]

if len(labelled) >= 10:
    y_true = (labelled["rating"] >= 4).map({True: "Positive", False: "Negative"})

    # 1) Standard BERT, one label per review
    evaluation["Standard BERT (whole review)"] = {
        "accuracy": accuracy_score(y_true, labelled["bert_label"]),
        "macro_f1": f1_score(y_true, labelled["bert_label"], average="macro"),
        "reviews": len(labelled)}

    # 2) ABSA rolled up: confidence-weighted average of the review's aspect scores
    rolled = mentions.groupby("review_id").apply(
        lambda x: np.average(x["score"], weights=x["confidence"] + 1e-6), include_groups=False)
    has_aspects = labelled["review_id"].isin(rolled.index)
    absa_pred = labelled.loc[has_aspects, "review_id"].map(rolled).map(lambda v: "Positive" if v > 0 else "Negative")
    evaluation["ABSA rolled up to review"] = {
        "accuracy": accuracy_score(y_true[has_aspects], absa_pred),
        "macro_f1": f1_score(y_true[has_aspects], absa_pred, average="macro"),
        "reviews": int(has_aspects.sum())}

    display(pd.DataFrame(evaluation).T.style.format({"accuracy": "{:.1%}", "macro_f1": "{:.3f}"}))

    # 3) Is confidence meaningful? Agreement with stars by confidence band
    polar = mentions.dropna(subset=["rating"])
    polar = polar[(polar["rating"] != 3) & (polar["sentiment"] != "Neutral")].copy()
    polar["agrees_with_stars"] = (polar["sentiment"] == "Positive") == (polar["rating"] >= 4)
    calibration = (polar.groupby("confidence_band", observed=False)["agrees_with_stars"]
                   .agg(["mean", "size"]).rename(columns={"mean": "agreement", "size": "mentions"}))
    evaluation["agreement_by_confidence"] = calibration.reset_index().to_dict("records")

    plt.figure(figsize=(7, 4))
    sns.barplot(x=calibration.index.astype(str), y=calibration["agreement"], color="#4C72B0")
    for i, (a, n) in enumerate(zip(calibration["agreement"], calibration["mentions"])):
        if n:
            plt.text(i, a + 0.02, f"{a:.0%}\nn={n}", ha="center")
    plt.ylim(0, 1.15)
    plt.ylabel("Aspect label agrees with stars")
    plt.xlabel("Confidence band")
    plt.title("Higher confidence should mean more agreement", fontweight="bold")
    plt.tight_layout()
    plt.show()

    print("💡 Aspect labels CAN legitimately disagree with stars: a 5★ review may still complain about the price.")
else:
    print("ℹ️ Not enough star ratings to evaluate.")

# 🖥️ STAGE 9 — Produce the dashboard

Everything is combined into **one HTML file**: summary numbers, strengths and weaknesses with real quotes,
all charts and the full aspect table. It opens in any browser, works offline, and is ready to submit or share.

In [ ]:
# CELL 24 — Build the HTML dashboard
# ============================================================

def fig_to_base64(fig):
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode()

def evidence_html(aspect, sentiment, k=2):
    rows = mentions[(mentions["aspect"] == aspect) & (mentions["sentiment"] == sentiment)]
    rows = rows.sort_values("confidence", ascending=False).drop_duplicates("sentence").head(k)
    out = ""
    for r in rows.itertuples():
        quote = html.escape(r.sentence)
        quote = re.sub(rf"\b({re.escape(html.escape(r.surface))})\b", r"<b>\1</b>", quote, count=1, flags=re.I)
        out += f'<q>{quote}</q><small>confidence {r.confidence:.2f}</small>'
    return out

def ranked_list_html(kind):
    col, sentiment, cls = (("pos_lower", "Positive", "pos") if kind == "strength"
                           else ("neg_lower", "Negative", "neg"))
    rows = summary.sort_values(col, ascending=False).head(6)
    rows = rows[rows[col] > 0]
    if rows.empty:
        return "<p>Not enough evidence yet.</p>"
    items = ""
    for aspect, r in rows.iterrows():
        share = r["positive_share"] if kind == "strength" else 1 - r["positive_share"]
        items += f"""<li><div class="row"><span class="name {cls}">{html.escape(aspect)}</span>
            <span>{share:.0%} of {int(r['positive'] + r['negative'])} opinions</span></div>
            <div class="bar"><i class="{cls}" style="width:{share * 100:.0f}%"></i></div>
            {evidence_html(aspect, sentiment)}</li>"""
    return f"<ol>{items}</ol>"

def build_dashboard(path):
    charts = []
    fig, ax = plt.subplots(figsize=(10, 0.45 * min(len(summary), 15) + 1.5)); plot_diverging(summary, ax=ax)
    charts.append(fig_to_base64(fig))
    fig, ax = plt.subplots(figsize=(10, 6)); plot_priority_map(summary, ax=ax)
    charts.append(fig_to_base64(fig))
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.histplot(data=mentions, x="confidence", hue="sentiment", palette=PALETTE, bins=25, multiple="stack", ax=ax)
    ax.set_title("Prediction confidence", fontweight="bold")
    charts.append(fig_to_base64(fig))
    if matrix.shape[0] >= 2 and matrix.notna().any().any():
        fig, ax = plt.subplots(figsize=(min(1.1 * matrix.shape[1] + 4, 16), 0.55 * matrix.shape[0] + 2))
        sns.heatmap(matrix, annot=True, fmt="+.2f", cmap="RdYlGn", vmin=-1, vmax=1, linewidths=0.5, ax=ax)
        ax.set_title("Aspect sentiment by product", fontweight="bold"); ax.set_ylabel(""); ax.set_xlabel("")
        charts.append(fig_to_base64(fig))

    n_pos = int((mentions["sentiment"] == "Positive").sum())
    n_neg = int((mentions["sentiment"] == "Negative").sum())
    high = (mentions["confidence_band"] == "High").mean()
    n_mixed = int(reviews["review_type"].str.startswith("Mixed").sum())

    table = summary[["mentions", "positive", "negative", "neutral", "positive_share",
                     "weighted_score", "avg_confidence", "verdict"]].copy()
    table["positive_share"] = table["positive_share"].map("{:.0%}".format)
    table["weighted_score"] = table["weighted_score"].map("{:+.2f}".format)
    table["avg_confidence"] = table["avg_confidence"].map("{:.2f}".format)

    eval_rows = "".join(
        f"<tr><td>{k}</td><td>{v['accuracy']:.1%}</td><td>{v['macro_f1']:.3f}</td><td>{v['reviews']}</td></tr>"
        for k, v in evaluation.items() if isinstance(v, dict))

    page = f"""<!doctype html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Product Review Intelligence Dashboard</title>
<style>
 body{{margin:0;background:#F5F6F8;color:#1F2A33;font:15px/1.5 "Segoe UI",system-ui,sans-serif}}
 main{{max-width:1150px;margin:auto;padding:28px 20px 50px}}
 h1{{margin:0;font-size:28px}} h2{{font-size:19px;margin:0 0 12px}}
 .sub{{color:#5F6B76;margin:4px 0 22px}}
 .kpis{{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:12px;margin-bottom:22px}}
 .kpi{{background:#fff;border-radius:10px;padding:14px 16px;border:1px solid #E3E6EA}}
 .kpi b{{display:block;font-size:26px}} .kpi span{{color:#5F6B76;font-size:13px}}
 .grid{{display:grid;grid-template-columns:1fr 1fr;gap:16px}} @media(max-width:820px){{.grid{{grid-template-columns:1fr}}}}
 .card{{background:#fff;border-radius:10px;padding:18px 20px;border:1px solid #E3E6EA;margin-bottom:16px}}
 ol{{list-style:none;margin:0;padding:0}} li{{padding:10px 0;border-top:1px solid #EEF0F2}}
 .row{{display:flex;justify-content:space-between;gap:10px;font-size:13px;color:#5F6B76}}
 .name{{font-weight:700;font-size:15px}} .name.pos{{color:{POS_COLOR}}} .name.neg{{color:{NEG_COLOR}}}
 .bar{{height:6px;background:#EEF0F2;border-radius:3px;margin:6px 0}} .bar i{{display:block;height:100%;border-radius:3px}}
 i.pos{{background:{POS_COLOR}}} i.neg{{background:{NEG_COLOR}}}
 q{{display:block;font-size:13.5px;color:#3A4651;margin-top:4px}} small{{color:#8A949C;font-size:11.5px}}
 img{{max-width:100%;display:block;margin:auto}}
 .wrap{{overflow-x:auto}} table{{border-collapse:collapse;width:100%;font-size:13.5px}}
 th,td{{padding:7px 10px;border-bottom:1px solid #EEF0F2;text-align:left;white-space:nowrap}} th{{background:#FAFBFC}}
</style></head><body><main>
<h1>🛒 Product Review Intelligence Dashboard</h1>
<p class="sub">{len(reviews):,} reviews from {html.escape(KAGGLE_DATASET)} · aspect sentiment by {html.escape(ABSA_MODEL)}</p>
<div class="kpis">
 <div class="kpi"><b>{len(summary)}</b><span>aspects tracked</span></div>
 <div class="kpi"><b>{len(mentions):,}</b><span>aspect opinions</span></div>
 <div class="kpi"><b style="color:{POS_COLOR}">{n_pos:,}</b><span>positive</span></div>
 <div class="kpi"><b style="color:{NEG_COLOR}">{n_neg:,}</b><span>negative</span></div>
 <div class="kpi"><b>{high:.0%}</b><span>high-confidence predictions</span></div>
 <div class="kpi"><b>{n_mixed}</b><span>mixed reviews 🍗</span></div>
</div>
<div class="grid">
 <div class="card"><h2>💪 Strengths</h2>{ranked_list_html("strength")}</div>
 <div class="card"><h2>⚠️ Weaknesses</h2>{ranked_list_html("weakness")}</div>
</div>
<div class="grid">{"".join(f'<div class="card"><img alt="chart" src="data:image/png;base64,{c}"></div>' for c in charts)}</div>
<div class="card"><h2>📋 All aspects</h2><div class="wrap">{table.to_html(border=0)}</div></div>
<div class="card"><h2>✅ Evaluation against star ratings</h2><div class="wrap"><table>
 <tr><th>Method</th><th>Accuracy</th><th>Macro-F1</th><th>Reviews</th></tr>{eval_rows}</table></div></div>
</main></body></html>"""
    with open(path, "w", encoding="utf-8") as f:
        f.write(page)
    return page

dashboard_path = os.path.join(OUTPUT_DIR, "dashboard.html")
dashboard_html = build_dashboard(dashboard_path)
print("✅ Dashboard saved to", dashboard_path)
display(HTML(dashboard_html))

# 🧪 STAGE 10 — Interactive demo: analyse any review

Type your own review. Aspects are found **automatically** with the Stage 3 rules, then each one gets a
sentiment and a confidence. Leave the box empty to use the example.

Example: `The laptop is fast but the battery is disappointing and the keyboard feels cheap.`

In [ ]:
# CELL 25 — Analyse your own review
# ============================================================

def analyse_review(text, extra_aspects=None):
    doc = nlp(clean_text(text))
    rows = extract_candidates(doc)
    for aspect in extra_aspects or []:                  # optional manual aspects
        for sent in doc.sents:
            if aspect.lower() in sent.text.lower():
                rows.append({"sent_idx": 0, "sentence": sent.text, "aspect": aspect, "surface": aspect})
    if not rows:
        print("No aspects found. Mention a product feature such as price, battery or screen.")
        return pd.DataFrame()
    df = pd.DataFrame(rows).drop_duplicates(["sentence", "aspect"])
    probs = predict_aspect_sentiment(df["sentence"].tolist(), df["surface"].tolist())
    df[["p_negative", "p_neutral", "p_positive"]] = probs[["Negative", "Neutral", "Positive"]].values
    df["sentiment"] = probs.idxmax(axis=1).values
    return add_confidence(df)

user_review = input("📝 Enter a review (or press Enter for the example): ").strip()
if not user_review:
    user_review = "The laptop is fast but the battery is disappointing and the keyboard feels cheap."

result = analyse_review(user_review)
if len(result):
    print("\nReview:", user_review)
    print("Standard BERT says:", document_pipeline(user_review)[0]["label"])
    display(result[["aspect", "sentiment", "confidence", "confidence_band", "p_negative", "p_neutral", "p_positive"]].round(3))

    plt.figure(figsize=(8, 0.6 * len(result) + 1.5))
    sns.barplot(data=result, y="aspect", x="confidence", hue="sentiment", palette=PALETTE, dodge=False)
    plt.xlim(0, 1)
    plt.title("Aspect sentiment and confidence", fontweight="bold")
    plt.tight_layout()
    plt.show()

# 🔁 STAGE 11 — The whole pipeline in one call

All stages above are reusable functions, so the complete system is just a few lines.
This is the version you would put in a project report or a Python script.

In [ ]:
# CELL 26 — One-call pipeline
# ============================================================

def run_review_intelligence(review_df):
    """review_df needs columns: review_id, text, rating, product."""
    review_docs = list(nlp.pipe(review_df["text"].tolist(), batch_size=64))           # Stage 2
    rows = [dict(r, review_id=rid) for rid, d in zip(review_df["review_id"], review_docs)
            for r in extract_candidates(d)]                                           # Stage 3
    cands = pd.DataFrame(rows)
    freq = cands.groupby("aspect")["review_id"].nunique()
    keep = freq[freq >= min(MIN_ASPECT_REVIEWS, max(2, len(review_df) // 20))].nlargest(MAX_ASPECTS).index
    m = cands[cands["aspect"].isin(keep)].merge(review_df[["review_id", "rating", "product"]], on="review_id")
    p = predict_aspect_sentiment(m["sentence"].tolist(), m["surface"].tolist())       # Stage 4
    m[["p_negative", "p_neutral", "p_positive"]] = p[["Negative", "Neutral", "Positive"]].values
    m["sentiment"] = p.idxmax(axis=1).values
    m = add_confidence(m)                                                             # Stage 5
    return m, summarise_aspects(m, min_mentions=2)                                    # Stage 6

# Quick test on the built-in demo reviews
demo_df = pd.DataFrame(DEMO_REVIEWS[:12], columns=["text", "rating", "product"]).reset_index(names="review_id")
demo_mentions, demo_summary = run_review_intelligence(demo_df)
display(demo_summary[["mentions", "positive", "negative", "positive_share", "weighted_score", "verdict"]])

In [ ]:
# CELL 27 — Save everything and download
# ============================================================

mentions.to_csv(os.path.join(OUTPUT_DIR, "aspect_mentions.csv"), index=False)
summary.to_csv(os.path.join(OUTPUT_DIR, "aspect_summary.csv"))
reviews.to_csv(os.path.join(OUTPUT_DIR, "reviews_with_predictions.csv"), index=False)
if matrix.notna().any().any():
    matrix.to_csv(os.path.join(OUTPUT_DIR, "product_aspect_matrix.csv"))
with open(os.path.join(OUTPUT_DIR, "evaluation.json"), "w") as f:
    json.dump({"dataset": KAGGLE_DATASET, "absa_model": ABSA_MODEL, "reviews": len(reviews),
               "evaluation": evaluation}, f, indent=2, default=float)

archive = shutil.make_archive("review_intelligence_outputs", "zip", OUTPUT_DIR)
print("✅ Saved:", sorted(os.listdir(OUTPUT_DIR)))

try:
    from google.colab import files
    files.download(archive)
except Exception:
    print("Download the zip from the Files panel on the left:", archive)

# 🎓 Mini Assessment

### Conceptual questions

1. Why do we send **sentences** rather than whole reviews to the ABSA model?
2. Why is "every noun is an aspect" not good enough on real data? Give two examples of noise.
3. Two predictions both have a top probability of 0.60. Why can one be less trustworthy than the other?
4. What does the Wilson lower bound protect us from when ranking aspects?
5. Why are star ratings only a *rough* check for aspect-level predictions?
6. Find one mixed review in Cell 22. Does the standard BERT label describe it well?

### Coding challenges

1. Add three synonyms to `ASPECT_SYNONYMS` that fit your dataset and rerun from Cell 10.
2. Change `KAGGLE_DATASET` to `"snap/amazon-fine-food-reviews"` — which aspects appear now?
3. Hand-label 50 rows of `aspect_mentions.csv` and compute the real aspect-level accuracy.
4. Try `BALANCE_RATINGS = False`. How does the weakness list change, and why?

# 🏁 Summary

```text
Kaggle reviews
   ↓  Stage 1  clean + sample
Sentences
   ↓  Stage 3  spaCy noun phrases + frequency filter
(sentence, aspect) pairs
   ↓  Stage 4  DeBERTa ABSA: sentence + aspect → Negative / Neutral / Positive
Probabilities
   ↓  Stage 5  probability + margin + certainty → confidence
Aspect predictions
   ↓  Stage 6  counts, weighted score, Wilson ranking
Strengths & weaknesses
   ↓  Stage 7–9 charts + HTML dashboard
```

**Big takeaway:** standard sentiment tells you *whether* customers are happy.
ABSA tells you *what* they are happy or unhappy about — and that is what a company can act on.

# ⚠️ Instructor Notes

**Execution order:** run Cell 1, then Cell 2 onward. No restart is needed because Cell 1 never re-installs
PyTorch or Transformers — it uses the versions Colab ships, which are built to work together.

**If you see `module 'torch' has no attribute '_utils'`** (or another torch import error): the session's packages
were changed while it was running. Use *Runtime → Disconnect and delete runtime*, then run from Cell 1.

**Runtime:** with a T4 GPU, 1,500 reviews take a few minutes end to end. On CPU keep `SAMPLE_SIZE` around 200.

**Models**

- Aspect sentiment: `yangheng/deberta-v3-base-absa-v1.1` (used with `text_pair=aspect`, as on its model card)
- Whole-review sentiment: `distilbert-base-uncased-finetuned-sst-2-english`

**Kaggle access:** `kagglehub` downloads public datasets without a login in most cases. If it asks for credentials,
either add `KAGGLE_USERNAME` / `KAGGLE_KEY` in Colab *Secrets*, or download the CSV manually and use Cell 5.

**Known limitations** (good discussion points)

- Noun-phrase extraction misses *implicit* aspects: "it's too heavy" is about weight but never says "weight".
- Star ratings are review-level, so they are a noisy reference for aspect-level labels.
- Model probabilities are not perfectly calibrated; Cell 23 checks this empirically.